<a href="https://colab.research.google.com/github/Shaithra2107/HALT-RAG/blob/lift_signal/notebooks/04_retrieval_and_hard_lift.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — Retrieval check and hard-negative Lift (Plan Steps 3 and 4)

Two parts, one corpus. The corpus is the **same one notebook 03 used**, rebuilt here
from the same cache with the same settings, and checked three ways before anything
is measured.

**Part 1 (Step 3) — did retrieval reach the right page?**
A question is a *hit* if any of its top-5 (and top-10) passages came from that
question's own subject page. Reported as Recall@5, Recall@10 and MRR@10, overall
and head vs tail, plus where the subject page's best passage actually ranks in the
whole collection.

**Part 2 — a harder background for Lift.**
`lift` uses a random sample of the corpus as its baseline. Random passages are easy
to beat, so `lift` may be measuring nothing more than "this question looks like a
Wikipedia question". `lift_hard` replaces that baseline with passages the retriever
*almost* chose:

```
lift_hard = (mean sim of top-5 - mean sim of ranks 21-100) / max(sd of ranks 21-100, eps)
gap_hard  =  mean sim of top-5 - mean sim of ranks 21-100          (no division)
score_gap =  top-1 sim - top-2 sim                                 (the usual cheap baseline)
```

`gap_hard` is `lift_hard` without the division, so the two together answer
"does dividing by the spread help?". `score_gap` is the standard baseline from the
query-performance-prediction literature, so the three together answer
"does any of this beat a one-line score gap?".

**Input** (Drive on Colab, `data\` on the laptop):

* `wiki2023_cache.pkl` — from `notebooks/02b_wiki2023_build.ipynb`
* `halt_rag_w23_signals.csv` and `corp_embs_w23.npy` — from `notebooks/03_signals_w23.ipynb`

**Output:** `halt_rag_w23_signals_hard.csv`, `halt_rag_w23_hard_results.json`

### This notebook never encodes anything

It loads `corp_embs_w23.npy` and refuses to run if the corpus it rebuilds does not
match it. Re-encoding would hide a mismatch instead of showing it — and a mismatch
means every passage index has shifted, so `lift` and `lift_hard` would be computed
over different collections and could not be compared at all.

### What ranks 21-100 are, and what they are not

They are **related passages, not passages that are known to be wrong.**

The question's own subject page is removed before the ranking is cut, so the gold
page can never appear in the window. But the collection also holds 5,000 other
Wikipedia pages, and nothing stops one of them from mentioning the same answer — a
list page, a "1998 deaths" page, a page about a related person. So the window means
"things the retriever rated highly but did not pick", **not** "things that are
definitely unhelpful".

State that as a limitation in the write-up. It makes `lift_hard` *conservative*: if
some of the background does contain the answer, the bar is set slightly too high and
Lift is slightly understated, not overstated.

### Still to do before the 1,000-question run

The corpus is built with a copy of notebook 03's inline `chunk_text`, on purpose:
this notebook has to reproduce 03's passage list exactly, index for index. **When 03
switches to `src/halt_rag` (`merge_into_passages`, `dedupe`), this notebook must
switch in the same commit**, or the integrity check below will start failing for a
good reason and you will have to re-run both.

## Cell 1 — Install

In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip install -q sentence-transformers==3.0.1 2>&1 | tail -2
    !pip install -q faiss-cpu
else:
    print("laptop: sentence-transformers and faiss-cpu come from .venv (requirements.txt)")
import torch; print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

## Cell 2 — Setup and settings

Every path, seed, size and window edge comes from `config.yaml`.

In [ ]:

# ── Setup: works on a laptop (VS Code) and on Colab ───────────────────────
import os, sys, json, subprocess
ON_COLAB = "google.colab" in sys.modules
if ON_COLAB:
    # Colab: get the code from GitHub (push your latest commit first) and mount Drive
    REPO, BRANCH = "/content/HALT-RAG", "lift_signal"
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                        "https://github.com/Shaithra2107/HALT-RAG", REPO], check=True)
    else:
        subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
else:
    # laptop: this notebook sits in <repo>/notebooks, so the repo is one level up
    REPO = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(REPO, "src"))

from halt_rag.config import load_config, data_path, data_dir
cfg = load_config(os.path.join(REPO, "config.yaml"))
print("running on:", "Colab" if ON_COLAB else "laptop", "| data folder:", data_dir(cfg))

# Nothing below is hard-coded: change config.yaml, not this cell.
CONFIG = {
    'audit_csv'    : data_path(cfg, "audit_w23"),          # halt_rag_audit_w23.csv
    'cache_pkl'    : data_path(cfg, "wiki2023_cache"),     # wiki2023_cache.pkl
    'signals_csv'  : data_path(cfg, "signals_w23"),        # written by notebook 03
    'emb_npy'      : data_path(cfg, "corpus_embs_w23"),    # written by notebook 03
    'out_signals'  : data_path(cfg, "signals_w23_hard"),   # halt_rag_w23_signals_hard.csv
    'out_results'  : data_path(cfg, "results_w23_hard"),

    'bi_encoder'   : cfg["retrieval"]["encoder"],
    'top_k'        : cfg["retrieval"]["top_k"],            # 5  — the list notebook 03 scored
    'top_k_wide'   : cfg["retrieval"]["top_k_wide"],       # 10 — Recall@10 and MRR@10
    'tie_tol'      : cfg["retrieval"]["tie_tol"],          # 1e-4 — scores this close are a tie
    'chunk_words'  : cfg["collection"]["passage_words"],
    'n_distractors': cfg["collection"]["background_pages_used"],
    'hard_start'   : cfg["signals"]["hard_neg_start"],     # 21
    'hard_end'     : cfg["signals"]["hard_neg_end"],       # 100
    'lift_eps'     : cfg["signals"]["lift_eps"],
    'n_boot'       : cfg["eval"]["n_bootstrap"],
    'alarm'        : cfg["eval"]["lift_relevance_alarm"],
    'seed'         : cfg["seed"],
}
print(json.dumps(CONFIG, indent=2))

for _k in ('audit_csv', 'cache_pkl', 'signals_csv', 'emb_npy'):
    assert os.path.exists(CONFIG[_k]), (
        f"Not found: {CONFIG[_k]}\n"
        f"Run notebooks/02b_wiki2023_build.ipynb and notebooks/03_signals_w23.ipynb first, "
        f"and copy their outputs into the data folder.")


## Cell 3 — Rebuild the same corpus as notebook 03

Same cache, same `chunk_words`, same `n_distractors`, same order: subject pages first
(in cache order), then the first N background pages. The chunker below is a copy of
notebook 03's, kept deliberately — see the note at the top.

While building, we also record which corpus rows came from which subject page. That
is what Part 1 needs: a question's "own page" rows.

In [ ]:
import pickle, re
import pandas as pd, numpy as np
from halt_rag.data.popqa import normalise_title   # the SAME key 02b used to build the cache

audit = pd.read_csv(CONFIG['audit_csv'])
sig   = pd.read_csv(CONFIG['signals_csv'])
print(f"audit   : {os.path.basename(CONFIG['audit_csv'])}  ({len(audit)} rows)")
print(f"signals : {os.path.basename(CONFIG['signals_csv'])}  ({len(sig)} rows)")
assert len(audit) == len(sig), "the audit and the 03 signals file have different lengths"
assert (audit['question'].values == sig['question'].values).all(), (
    "the 03 signals file is not in the same row order as the audit — "
    "re-run notebook 03, it reads the audit in order")

with open(CONFIG['cache_pkl'], 'rb') as f:
    _c = pickle.load(f)
kilt_pages      = _c['kilt_pages']
distractor_pool = _c['distractor_pool']
print(f"cache   : {os.path.basename(CONFIG['cache_pkl'])} | source: {_c.get('source', 'unknown')}")
print(f"subject pages {len(kilt_pages)} | background pages {len(distractor_pool)}")

def chunk_text(paragraphs, w=CONFIG['chunk_words']):
    out = []
    for para in paragraphs:
        words = str(para).split()
        for i in range(0, len(words), w):
            c = ' '.join(words[i:i+w])
            if len(c.strip()) > 30: out.append(c)
    return out

passages, titles, is_entity = [], [], []
page_rows = {}                      # normalised subject title -> row numbers in the corpus
for norm_title, paras in kilt_pages.items():
    rows = []
    for c in chunk_text(paras):
        rows.append(len(passages))
        passages.append(c); titles.append(norm_title); is_entity.append(1)
    if rows:
        page_rows[norm_title] = np.array(rows, dtype=np.int64)
for d in distractor_pool[:CONFIG['n_distractors']]:
    for c in chunk_text(d['paragraphs']):
        passages.append(c); titles.append(d['title']); is_entity.append(0)

is_entity = np.array(is_entity)
print(f"\ncorpus: {len(passages):,} passages "
      f"({is_entity.sum():,} from subject pages, {(1-is_entity).sum():,} background)")
print(f"subject pages with at least one passage: {len(page_rows)}")


## Cell 4 — Load the cached embeddings and prove the corpus is the same

Three checks, strongest last:

1. the meta file written by 03 matches (passage count, chunk size, encoder);
2. the embedding matrix has one row per passage;
3. **the top-5 this notebook retrieves equals the `ret_idx` column notebook 03 saved.**

Check 3 is the one that matters. If a single passage had been added or dropped, every
index after it would shift and check 3 would fail even when 1 and 2 passed.

**GPU vs CPU rounding.** Notebook 03 may have run on a GPU and notebook 04 on a CPU (or
vice versa). Float32 arithmetic differs at the last bit, so two passages that are
nearly tied at rank 5/6 can swap. If that happens for a question, the check prints the
two passages and their scores. If the difference is within `tie_tol` (1e-4 by default,
set in `config.yaml`), the swap is treated as a tie and notebook 03's saved ordering is
adopted for that question — the two passages are interchangeable in everything below.
If the difference is larger the check stops with an error: the corpus or the encoder
has drifted for real.


In [ ]:

from sentence_transformers import SentenceTransformer
from halt_rag.retrieval import dense

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
bi  = SentenceTransformer(CONFIG['bi_encoder'], device=DEV)

EMB_NPY, EMB_META = CONFIG['emb_npy'], CONFIG['emb_npy'] + '.meta.json'
assert os.path.exists(EMB_META), (
    f"No meta file beside {EMB_NPY}. It is written by notebook 03 — copy it over too.")
meta = json.load(open(EMB_META))
assert meta['n_passages'] == len(passages), (
    f"CHECK 1 FAILED: the cache has {meta['n_passages']:,} passages, this corpus has "
    f"{len(passages):,}. The corpus changed since notebook 03 ran. Do NOT re-encode here — "
    f"re-run notebook 03 so both notebooks share one corpus.")
assert meta['chunk_words'] == CONFIG['chunk_words'] and meta['bi_encoder'] == CONFIG['bi_encoder'], (
    f"CHECK 1 FAILED: settings differ. cache={meta}, here chunk_words={CONFIG['chunk_words']}, "
    f"encoder={CONFIG['bi_encoder']}")

corp_embs = np.load(EMB_NPY)
assert len(corp_embs) == len(passages), "CHECK 2 FAILED: embeddings do not match passages"
print(f"loaded {corp_embs.shape[0]:,} x {corp_embs.shape[1]} cached embeddings (no encoding done)")

index  = dense.build_index(corp_embs)
q_emb  = dense.encode(bi, audit['question'].tolist(), batch_size=128)
K_WIDE = CONFIG['top_k_wide']
ret_scores, ret_ids = dense.search(index, q_emb, K_WIDE)
print(f"index: {index.ntotal:,} vectors | retrieved top-{K_WIDE}: {ret_ids.shape}")

# CHECK 3 — the same top-5 notebook 03 scored; GPU/CPU rounding may swap tied passages
import ast
TOP_K   = CONFIG['top_k']
TIE_TOL = CONFIG['tie_tol']
old_top5 = np.array([ast.literal_eval(r) for r in sig['ret_idx']], dtype=np.int64)
new_top5 = ret_ids[:, :TOP_K].astype(np.int64)
same_set  = [set(a) == set(b) for a, b in zip(old_top5, new_top5)]
n_mismatch = len(same_set) - sum(same_set)

if n_mismatch == 0:
    print(f"CHECK 3 ok: all {len(same_set)} questions retrieve exactly 03's top-5")
else:
    print(f"CHECK 3: {n_mismatch} question(s) differ — checking scores for ties "
          f"(tie_tol={TIE_TOL}) ...")
    for i in [i for i, ok in enumerate(same_set) if not ok]:
        old_set = set(old_top5[i].tolist())
        new_set = set(new_top5[i].tolist())
        dropped = sorted(old_set - new_set)   # in 03's top-5 but not ours
        added   = sorted(new_set - old_set)   # in our top-5 but not 03's
        threshold = float(ret_scores[i, TOP_K - 1])   # our rank-5 score
        print(f"\n  q[{i}]: {audit['question'].iloc[i]!r}")
        print(f"  rank-5 score threshold: {threshold:.6f}  tie_tol: {TIE_TOL}")
        for pid in dropped + added:
            pos = np.flatnonzero(ret_ids[i] == pid)
            if len(pos):
                sc = float(ret_scores[i, pos[0]])
            else:
                sc = float(corp_embs[pid] @ q_emb[i])   # outside the K_WIDE list
            tag  = "03 only" if pid in old_set - new_set else "ours only"
            diff = abs(sc - threshold)
            print(f"    passage {pid:6d} ({tag}): score {sc:.6f}  |diff| {diff:.2e}")
            if diff > TIE_TOL:
                raise AssertionError(
                    f"CHECK 3 FAILED (not a tie): q[{i}] {audit['question'].iloc[i]!r} — "
                    f"passage {pid} ({tag}) scores {sc:.6f}, rank-5 threshold is "
                    f"{threshold:.6f}, difference {diff:.2e} > tie_tol {TIE_TOL}. "
                    f"The corpus or the encoder has drifted — stop here and re-run "
                    f"notebook 03 so both notebooks use the same retrieval results.")
        # confirmed tie — adopt 03's ordering (the two passages are interchangeable).
        # ret_ids[i, 5] will then be a duplicate of old_top5[i, 4]; that is harmless
        # because recall_at_k and mrr both use set operations, not positional indexing.
        ret_ids[i, :TOP_K] = old_top5[i]
        print(f"  -> tie confirmed: using notebook 03's ret_idx for q[{i}]")
    print(f"\nCHECK 3 ok: {n_mismatch} tie(s) resolved — notebook 03's ordering adopted")

same_order = int((old_top5 == ret_ids[:, :TOP_K]).all(axis=1).sum())
print(f"({same_order} of {len(same_set)} questions in exactly 03's order after tie resolution)")


## Cell 5 — Part 1: each question's own subject page

A question's gold set is the corpus rows that came from its subject page, keyed by
`normalise_title(wiki_title)` — imported from `halt_rag.data.popqa`, which is the same
function `02b_wiki2023_build.ipynb` used to build the cache, so the key cannot drift.

One question has no page at all (`m2_status_w23 == page_missing`). It has no gold set,
so it is **dropped from Recall and MRR and reported as dropped** — scoring it as a miss
would blame the retriever for a page that is not in the collection.

In [ ]:
own_rows = [page_rows.get(normalise_title(t), np.array([], dtype=np.int64))
            for t in audit['wiki_title']]
gold_sets = [set(map(int, r)) for r in own_rows]
has_page  = np.array([len(r) > 0 for r in own_rows])

print(f"questions with a subject page in the corpus: {has_page.sum()} / {len(audit)}")
print(f"dropped (no page): {(~has_page).sum()}")
if (~has_page).any():
    print(audit.loc[~has_page, ['question', 'wiki_title', 'm2_status_w23']].to_string(index=False))

print("\npage status (this is about the ANSWER, not about retrieval):")
print(audit['m2_status_w23'].value_counts().to_string())
print("\nNOTE: 'found' means the answer is on the subject page; 'answer_not_on_page' means")
print("the page is in the collection but the answer is not written on it. Part 1 measures")
print("whether retrieval REACHED the page, which is a different question from both.")
print(f"passages per subject page: median "
      f"{int(np.median([len(r) for r in own_rows if len(r)]))}, "
      f"max {max(len(r) for r in own_rows)}")

## Cell 6 — Recall@5, Recall@10, MRR@10 — overall and head vs tail

`recall_at_k` and `mrr` come from `src/halt_rag/retrieval/dense.py`, so these numbers
are produced by the package, not by a second copy of the formula in a notebook.

MRR is computed over the top-10 list, so a page first seen at rank 11 or later scores
0. That is why it is labelled **MRR@10** everywhere, including in the write-up.

In [ ]:
TOP_K = CONFIG['top_k']
hit5  = dense.recall_at_k(ret_ids[:, :TOP_K], gold_sets)
hit10 = dense.recall_at_k(ret_ids[:, :K_WIDE], gold_sets)
rr10  = dense.mrr(ret_ids[:, :K_WIDE], gold_sets)

def retrieval_row(name, mask):
    m = mask & has_page
    return {'scope': name, 'n': int(m.sum()),
            f'Recall@{TOP_K}': float(hit5[m].mean()), f'Recall@{K_WIDE}': float(hit10[m].mean()),
            f'MRR@{K_WIDE}': float(rr10[m].mean())}

band = audit['band'].astype(str).str.lower().values
part1 = pd.DataFrame([retrieval_row('overall', np.ones(len(audit), bool)),
                      retrieval_row('head', band == 'head'),
                      retrieval_row('tail', band == 'tail')]).set_index('scope')
display(part1.round(3))

d = part1.loc['head', f'Recall@{TOP_K}'] - part1.loc['tail', f'Recall@{TOP_K}']
print(f"head - tail on Recall@{TOP_K}: {d:+.3f}")
print("A large positive gap means retrieval itself is popularity-biased, before any")
print("signal is computed. That belongs in the Objective 3 discussion, not just here.")

## Cell 7 — Where the subject page's first passage ranks

Recall@10 only says "inside the top 10 or not". This cell gives the exact rank of the
page's **best** passage against the whole collection, by scoring every passage. Rank 1
means the page won; a rank in the thousands means retrieval never came close.

In [ ]:
from tqdm.auto import tqdm

subj_rank = np.full(len(audit), np.nan)
for i in tqdm(range(len(audit)), desc='subject-page rank'):
    if not has_page[i]:
        continue
    s = corp_embs @ q_emb[i]                       # exact: the index is flat inner product
    best = float(s[own_rows[i]].max())
    subj_rank[i] = int((s > best).sum()) + 1       # how many passages beat the page's best one

ranked = subj_rank[has_page]
print(f"rank of the subject page's best passage (n={len(ranked)}, out of {len(passages):,}):")
print(f"  median {np.median(ranked):,.0f} | 25th {np.percentile(ranked, 25):,.0f} "
      f"| 75th {np.percentile(ranked, 75):,.0f} | worst {ranked.max():,.0f}")

EDGES = [1, TOP_K, K_WIDE, 100, 1000]
def buckets(mask):
    r = subj_rank[mask & has_page]
    out = {'n': len(r)}
    for e in EDGES:
        out[f'<={e}'] = float((r <= e).mean()) if len(r) else np.nan
    out[f'>{EDGES[-1]}'] = float((r > EDGES[-1]).mean()) if len(r) else np.nan
    return out

rank_tab = pd.DataFrame({'overall': buckets(np.ones(len(audit), bool)),
                         'head': buckets(band == 'head'),
                         'tail': buckets(band == 'tail')}).T
display(rank_tab.round(3))
print("Each column is the share of questions whose subject page reached that rank or better.")

## Cell 8 — Part 2: `lift_hard`, `gap_hard` and `score_gap`

For every question:

1. score every passage in the collection;
2. **remove the question's own subject page first**, then rank what is left;
3. take positions 21-100 of that ranking — exactly 80 passages for every question;
4. `lift_hard = (mean top-5 sim - mean of those 80) / max(sd of those 80, eps)`.

Removing the page *before* cutting the window matters. If we cut first and removed
afterwards, a question whose page retrieves well would lose more of its window and end
up with a smaller, noisier background — retrieval quality would leak into the
denominator, which is the exact mistake the pilot's Lift already made once.

The numerator keeps notebook 03's top-5 untouched, so `lift` and `lift_hard` differ in
their background and in nothing else.

**Again: ranks 21-100 are related passages, not known-wrong ones.** Other pages in the
collection may mention the same answer, which makes `lift_hard` conservative rather
than optimistic.

In [ ]:
HARD_START, HARD_END = CONFIG['hard_start'], CONFIG['hard_end']
EPS, WIDTH = CONFIG['lift_eps'], CONFIG['hard_end'] - CONFIG['hard_start'] + 1
assert 1 < HARD_START <= HARD_END < len(passages), \
    f"the {HARD_START}-{HARD_END} window does not fit a corpus of {len(passages):,}"
print(f"hard background: ranks {HARD_START}-{HARD_END} ({WIDTH} passages per question), "
      f"subject page removed first")

lift_hard, gap_hard, hard_mu, hard_sd, rel_new = [], [], [], [], []
for i in tqdm(range(len(audit)), desc='hard background'):
    s    = corp_embs @ q_emb[i]
    rel  = float(s[ret_ids[i, :TOP_K]].mean())        # same top-5 as notebook 03
    rel_new.append(rel)

    s2 = s.copy()
    if has_page[i]:
        s2[own_rows[i]] = -np.inf                      # option (i): drop the page, THEN rank
    part = np.argpartition(-s2, HARD_END)[:HARD_END]
    part = part[np.argsort(-s2[part])]
    win  = part[HARD_START - 1:HARD_END]
    assert len(win) == WIDTH and np.isfinite(s2[win]).all()

    hs = s2[win]
    mu, sd = float(hs.mean()), float(hs.std())
    hard_mu.append(mu); hard_sd.append(sd)
    gap_hard.append(rel - mu)
    lift_hard.append((rel - mu) / max(sd, EPS))

# a fourth integrity check: our relevance must equal the one notebook 03 saved
drift = float(np.abs(np.array(rel_new) - sig['relevance'].values).max())
assert drift < 1e-5, f"relevance differs from notebook 03 by {drift:.2e} — the corpus drifted"
print(f"relevance matches notebook 03 (largest difference {drift:.1e})")

score_gap = ret_scores[:, 0] - ret_scores[:, 1]        # the cheap baseline: top-1 minus top-2

hard = sig.copy()
hard['score_gap'] = score_gap
hard['gap_hard']  = gap_hard
hard['lift_hard'] = lift_hard
hard['hard_mu'], hard['hard_sd'] = hard_mu, hard_sd
hard['subj_rank'] = subj_rank
hard['hit5'], hard['hit10'], hard['rr10'] = hit5, hit10, rr10
hard['has_page'] = has_page

cols = ['lift', 'lift_hard', 'relevance', 'score_gap', 'gap_hard', 'rand_mu', 'hard_mu', 'hard_sd']
print()
print(hard[cols].describe().round(3).to_string())
print("\nhard_mu should sit well ABOVE rand_mu: the hard background is passages the")
print("retriever liked, the random one is passages it never saw.")
print(f"  mean rand_mu {hard['rand_mu'].mean():.3f}  ->  mean hard_mu {hard['hard_mu'].mean():.3f}")
print(f"  mean rand_sd {hard['rand_sd'].mean():.3f}  ->  mean hard_sd {hard['hard_sd'].mean():.3f}")

## Cell 9 — Save the signals file

Every column of `halt_rag_w23_signals.csv`, plus the new ones. Notebook 03's file is
not touched, so both can be read side by side.

In [ ]:
hard.to_csv(CONFIG['out_signals'], index=False)
print("saved", CONFIG['out_signals'], f"({len(hard)} rows, {len(hard.columns)} columns)")
print("new columns:", [c for c in hard.columns if c not in sig.columns])

## Cell 10 — AUROC with 95% ranges, and the popularity AUROC

`signal_table` from `src/halt_rag/eval/metrics.py` — the same function notebook 01
uses, so these numbers are directly comparable with the Step 1 ones.

`popularity_detector` is True when a signal predicts head/tail better than it predicts
the label, and its popularity range excludes 0.50. That flag is the Objective 3
question in one column: a flagged signal is measuring how famous the entity is.

In [ ]:
from halt_rag.eval.metrics import signal_table, auroc_ci

NEW     = ['lift', 'lift_hard', 'relevance', 'score_gap', 'gap_hard']
targets = [t for t in ['y_judge', 'y_cov', 'y_gen'] if t in hard.columns]
n_boot, seed = CONFIG['n_boot'], CONFIG['seed']

scopes = {'all': hard}
if 'm2_status_w23' in hard.columns:
    scopes['evaluable'] = hard[hard['m2_status_w23'].isin(['found', 'answer_not_on_page'])]
    print("evaluable scope from: m2_status_w23 (found / answer_not_on_page)")
else:
    print("no m2_status_w23 column -> 'evaluable' scope skipped")

tables = {}
for scope, d in scopes.items():
    for t in targets:
        tab = signal_table(d, NEW, t, n_boot=n_boot, seed=seed)
        tables[f"{scope}|{t}"] = tab
        print(f"\n=== scope={scope} (n={len(d)}) | label={t} (pos={int(d[t].sum())}) ===")
        display(tab.round(3))

## Cell 11 — The gain test, exactly as in notebook 01

Out-of-fold logistic regression, 5 stratified folds, scaler inside the pipeline, seed
from `config.yaml`. Every row is "relevance, plus one more thing" against relevance on
its own, so the `gain` column is the only number that matters: it is what the extra
signal is worth **on top of** relevance, not on its own.

The decision rule is the Step 1 rule, unchanged: warn if the gain is at or below 0.01
**or** the correlation with relevance is above the alarm.

In [ ]:
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

corr = {}
for s in ['lift', 'lift_hard', 'score_gap', 'gap_hard']:
    rho = spearmanr(hard[s], hard['relevance']).correlation
    corr[s] = float(rho)
    print(f"corr({s:9s}, relevance): Spearman {rho:+.3f} | "
          f"Pearson {pearsonr(hard[s], hard['relevance'])[0]:+.3f}")
print(f"\ncorr(lift, lift_hard): Spearman {spearmanr(hard['lift'], hard['lift_hard']).correlation:+.3f}")

def oof_auc(cols, target, d):
    """Out-of-fold AUROC of a logistic regression on these columns."""
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    p = cross_val_predict(m, d[cols], d[target], cv=cv, method="predict_proba")[:, 1]
    return auroc_ci(d[target], p, n_boot, seed=seed)

COMBOS = {
    'relevance + lift'            : ['relevance', 'lift'],
    'relevance + lift_hard'       : ['relevance', 'lift_hard'],
    'relevance + score_gap'       : ['relevance', 'score_gap'],
    'relevance + gap_hard'        : ['relevance', 'gap_hard'],
    'relevance + lift + lift_hard': ['relevance', 'lift', 'lift_hard'],
}
rows = []
for t in targets:
    base = oof_auc(['relevance'], t, hard)[0]
    for name, feats in COMBOS.items():
        a = oof_auc(feats, t, hard)[0]
        rows.append({'label': t, 'model': name, 'relevance_only': base,
                     'with_extra': a, 'gain': a - base})
added = pd.DataFrame(rows).set_index(['label', 'model'])
display(added.round(3))

## Cell 12 — Decision

Same wording as notebook 01, now applied to `lift_hard`. The question this notebook was
built to answer is on the last two lines: does the hard background beat the random one,
and does either beat a plain score gap?

In [ ]:
alarm, main = CONFIG['alarm'], targets[0]
gain_hard  = float(added.loc[(main, 'relevance + lift_hard'), 'gain'])
too_close  = bool(abs(corr['lift_hard']) > alarm)
adds_nothing = bool(gain_hard <= 0.01)

print(f"corr(lift_hard, relevance) = {corr['lift_hard']:.3f}  (alarm above {alarm})"
      f"  -> {'FAIL' if too_close else 'ok'}")
print(f"AUROC gain from adding lift_hard ({main}) = {gain_hard:+.3f}  (alarm at or below +0.010)"
      f"  -> {'FAIL' if adds_nothing else 'ok'}")

if too_close or adds_nothing:
    why = []
    if too_close:
        why.append(f"it moves with relevance (Spearman {corr['lift_hard']:.3f})")
    if adds_nothing:
        why.append(f"it adds no AUROC on top of relevance ({gain_hard:+.3f})")
    print("\nDECISION: lift_hard ~ relevance, because " + " and ".join(why) + ".")
    print("A harder background did not rescue Lift. Report that as a negative result.")
else:
    print("\nDECISION: lift_hard is distinct and adds something relevance does not.")
    print("Report lift and lift_hard as two variants, with this comparison as the evidence.")

cmp = {k: float(added.loc[(main, k), 'gain']) for k in COMBOS}
print(f"\nGains on {main}, best first:")
for k, v in sorted(cmp.items(), key=lambda kv: -kv[1]):
    print(f"  {k:30s} {v:+.3f}")
print(f"\nhard background vs random background : {cmp['relevance + lift_hard'] - cmp['relevance + lift']:+.3f}")
print(f"dividing by the spread is worth       : {cmp['relevance + lift_hard'] - cmp['relevance + gap_hard']:+.3f}")
print(f"lift_hard vs the plain score gap      : {cmp['relevance + lift_hard'] - cmp['relevance + score_gap']:+.3f}")
print("\nAll three are single-run differences on 300 questions. Read them next to the 95%")
print("ranges in Cell 10 before claiming any of them is real.")

## Cell 13 — Save the results

In [ ]:
out = {
    'notebook'     : '04_retrieval_and_hard_lift',
    'config'       : CONFIG,
    'corpus_passages': len(passages),
    'subject_passages': int(is_entity.sum()),
    'part1_retrieval': part1.reset_index().to_dict('records'),
    'subject_rank_buckets': rank_tab.reset_index().rename(columns={'index': 'scope'})
                            .to_dict('records'),
    'subject_rank_summary': {'n': int(has_page.sum()),
                             'median': float(np.median(ranked)),
                             'p25': float(np.percentile(ranked, 25)),
                             'p75': float(np.percentile(ranked, 75)),
                             'worst': float(ranked.max()),
                             'dropped_no_page': int((~has_page).sum())},
    'corr_with_relevance_spearman': corr,
    'gain_table'   : added.round(4).reset_index().to_dict('records'),
    'decision'     : {'rule': 'warn if gain <= 0.01 OR |spearman| > alarm',
                      'signal': 'lift_hard', 'label': main, 'alarm': alarm,
                      'gain': gain_hard, 'too_close': too_close,
                      'adds_nothing': adds_nothing, 'warn': too_close or adds_nothing},
    'signal_tables': {k: v.round(4).to_dict() for k, v in tables.items()},
}
json.dump(out, open(CONFIG['out_results'], 'w'), indent=2, default=float)
print("saved:", CONFIG['out_signals'])
print("saved:", CONFIG['out_results'])
print("\nIMPORTANT: File > Download > .ipynb WITH OUTPUTS, and copy both files back")
print("into data/ on the laptop, or these numbers are gone when the Colab session ends.")

---

### What this notebook does not answer

1. **Part 1 is about pages, not answers.** A hit means the subject page was retrieved.
   38 of the 299 questions have a page that does not contain the answer, so a high
   Recall@5 can sit next to hard labels. Never quote one without the other.
2. **Ranks 21-100 may contain the answer.** The window is "related", not "wrong" — see
   the note at the top. `lift_hard` is conservative because of it.
3. **The labels are still provisional.** `y_judge` uses an invented prompt and `y_cov`
   is a string match. Everything in Cells 10-12 inherits that, exactly as in notebook 01.
4. **300 questions, one split, one seed.** The gain differences in Cell 12 are small
   numbers from one run. The 1,000-question study is what settles them.